> **ARCHIVED research notebook** - kept for provenance and understanding only.
> It is **superseded by the `cloaq` package** (`src/cloaq/`) and may contain known bugs; see [`research/legacy-notebooks/README.md`](../README.md).
> Saved outputs were cleared and local paths removed. QASMBench paths are relative to your own checkout.

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.circuit.library import Barrier, CXGate
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import RemoveBarriers
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
from math import pi
import random
import numpy as np

backend = AerSimulator()

In [ ]:
def transform_circuit(qc, seed=42, verbose=True):
    if seed is not None:
        random.seed(seed)

    phase_gates = {'rz', 'p', 's', 't', 'sdg', 'tdg'}

    orig_instrs = [
        inst for inst in qc.data
        if not isinstance(inst.operation, Barrier)
    ]
    m = len(orig_instrs)

    n = min(6, m)
    key = [random.randint(0, 1) for _ in range(n)]
    n1 = sum(key)
    real_gate_indices = sorted(random.sample(range(m), n1))

    key_qreg = QuantumRegister(1, 'qk')
    locked_qc = QuantumCircuit(*qc.qregs, key_qreg, *qc.cregs)
    qk = key_qreg[0]

    key_ptr = 0
    real_ptr = 0
    last_phase = None
    qk_in_h_basis = False
    has_emitted_qll_cx = False

    for idx, inst in enumerate(orig_instrs):
        is_phase = inst.operation.name.lower() in phase_gates

        if last_phase is not None and is_phase != last_phase:
            locked_qc.barrier(*locked_qc.qubits)

        if key_ptr < n:
            key_bit = key[key_ptr]
            target = inst.qubits[0]

            if (key_bit == 1 and real_ptr < n1 and idx == real_gate_indices[real_ptr]):
                if not qk_in_h_basis:
                    locked_qc.h(qk)
                    locked_qc.barrier(qk)
                    qk_in_h_basis = True

                locked_qc.cx(qk, target)

                if has_emitted_qll_cx:
                    locked_qc.x(qk)
                    qk_in_h_basis = False
                else:
                    qk_in_h_basis = False
                    has_emitted_qll_cx = True

                real_ptr += 1
                key_ptr += 1

            elif key_bit == 0:
                if not qk_in_h_basis:
                    locked_qc.h(qk)
                    locked_qc.barrier(qk)
                    qk_in_h_basis = True

                locked_qc.cx(qk, target)

                if has_emitted_qll_cx:
                    locked_qc.x(qk)
                    qk_in_h_basis = False
                else:
                    qk_in_h_basis = False

                key_ptr += 1

        locked_qc.append(inst.operation, inst.qubits, inst.clbits)

        if inst.operation.name.lower() == "h":
            locked_qc.barrier(*inst.qubits)

        last_phase = is_phase

    if last_phase:
        locked_qc.barrier(*locked_qc.qubits)

    if verbose:
        print("---- Integrated QLL + Phase-Wall Locking ----")
        print(f"Key            : {key}")
        print(f"Real gates     : {real_gate_indices}")
        print(f"Original gates : {m}")
        print(f"Locked gates   : {len(locked_qc.data)}")

    return locked_qc, key, {"real_gate_indices": real_gate_indices}

In [ ]:
def modify_rz_inside_barriers(circuit: QuantumCircuit, phase_shift: float):
    """Adds a phase shift to all RZ gates inside barrier regions."""
    new_circuit = QuantumCircuit(*circuit.qregs, *circuit.cregs)
    qubit_map = {q: new_circuit.qubits[i] for i, q in enumerate(circuit.qubits)}
    clbit_map = {old: new_circuit.clbits[i] for i, old in enumerate(circuit.clbits)}
    inside_barrier = False

    for inst in circuit.data:
        op = inst.operation
        qargs = inst.qubits
        cargs = inst.clbits

        if op.name == 'barrier':
            inside_barrier = not inside_barrier
            new_circuit.barrier(*[qubit_map[q] for q in qargs])
            continue

        if inside_barrier and op.name == 'rz':
            new_circuit.rz(op.params[0] + phase_shift, qubit_map[qargs[0]])
        else:
            new_circuit.append(op, [qubit_map[q] for q in qargs],
                               [clbit_map[c] for c in cargs])

    return new_circuit


def subtract_phase_offset(circuit: QuantumCircuit, offset: float) -> QuantumCircuit:
    """Subtracts phase offset from RZ/P gates inside barriers."""
    new_circuit = QuantumCircuit(*circuit.qregs, *circuit.cregs)
    inside_barrier = False

    for instruction, qargs, cargs in circuit.data:
        if instruction.name == 'barrier':
            inside_barrier = not inside_barrier
            new_circuit.append(instruction, qargs, cargs)
            continue

        if inside_barrier and instruction.name in ['rz', 'p']:
            theta = instruction.params[0]
            if instruction.name == 'rz':
                new_circuit.rz(theta - offset, qargs[0])
            else:
                new_circuit.p(theta - offset, qargs[0])
            continue

        new_circuit.append(instruction, qargs, cargs)

    return new_circuit


def decrypt_qll(locked_circuit: QuantumCircuit, qll_key) -> QuantumCircuit:
    """
    QLL Decryption:
    - Drops all H gates on qk
    - Drops ALL CX gates controlled by qk (both real and dummy)
    - No X compensation needed — QLL CX gates are purely additive
    - Passes all original gates through unchanged
    """
    qk = locked_circuit.qregs[-1][0]
    qc = QuantumCircuit(*locked_circuit.qregs, *locked_circuit.cregs)
    qubit_map = {q: qc.qubits[i] for i, q in enumerate(locked_circuit.qubits)}
    clbit_map = {old: qc.clbits[i] for i, old in enumerate(locked_circuit.clbits)}
    key_ptr = 0

    for instruction in locked_circuit.data:
        op = instruction.operation
        qargs = instruction.qubits
        cargs = instruction.clbits

        if op.name == "barrier":
            continue

        if op.name == "h" and qk in qargs:
            continue

        if op.name == "cx" and qargs[0] == qk:
            if key_ptr < len(qll_key):
                print(f"QLL CX #{key_ptr}: key_bit={qll_key[key_ptr]}, "
                      f"target={qargs[1]} → dropped")
                key_ptr += 1
            continue

        if qk in qargs:
            continue

        new_qargs = [qubit_map[q] for q in qargs]
        new_cargs = [clbit_map[c] for c in cargs]
        qc.append(op, new_qargs, new_cargs)

    return qc


def decrypt_full_circuit(locked_circuit: QuantumCircuit, qll_key, phase_offset: float):
    """Fully decrypts a QLL + Phase-Wall locked circuit."""
    # Step 1: QLL unlock
    qc = decrypt_qll(locked_circuit, qll_key)

    # Step 2: Phase-wall unlock
    qc = subtract_phase_offset(qc, phase_offset)

    # Step 3: Barrier cleanup
    pm = PassManager(RemoveBarriers())
    qc = pm.run(qc)

    # Step 4: Strip qk register
    qk = qc.qregs[-1][0]
    new_qregs = [reg for reg in qc.qregs if reg.name != 'qk']
    new_circuit = QuantumCircuit(*new_qregs, *qc.cregs)
    qubit_map = {q: new_circuit.qubits[i]
                 for i, q in enumerate(qc.qubits) if q != qk}
    clbit_map = {old: new_circuit.clbits[i]
                 for i, old in enumerate(qc.clbits)}

    for inst in qc.data:
        if qk in inst.qubits:
            continue
        new_qargs = [qubit_map[q] for q in inst.qubits]
        new_cargs = [clbit_map[c] for c in inst.clbits]
        new_circuit.append(inst.operation, new_qargs, new_cargs)

    return new_circuit

In [ ]:
def remove_all_measurements(circuit):
    new_qregs = [QuantumRegister(reg.size, reg.name) for reg in circuit.qregs]
    new_cregs = [ClassicalRegister(reg.size, reg.name) for reg in circuit.cregs]
    new_circuit = QuantumCircuit(*new_qregs, *new_cregs)

    qubit_map = {old_reg[i]: new_qregs[j][i]
                 for j, old_reg in enumerate(circuit.qregs)
                 for i in range(old_reg.size)}
    clbit_map = {old_reg[i]: new_cregs[j][i]
                 for j, old_reg in enumerate(circuit.cregs)
                 for i in range(old_reg.size)}

    for instruction in circuit.data:
        if instruction.operation.name == 'measure':
            continue
        new_circuit.append(instruction.operation,
                           [qubit_map[q] for q in instruction.qubits],
                           [clbit_map[c] for c in instruction.clbits])
    return new_circuit


def get_measurement_info(circuit, exclude_qk=True):
    measurement_info = {}
    qk_index = None
    if exclude_qk and len(circuit.qregs) > 0:
        qk_index = circuit.qubits.index(circuit.qregs[-1][0])

    for instruction in circuit.data:
        if instruction.operation.name == 'measure':
            qubit_index = circuit.find_bit(instruction.qubits[0]).index
            clbit_index = circuit.find_bit(instruction.clbits[0]).index
            if exclude_qk and qubit_index == qk_index:
                continue
            measurement_info[qubit_index] = {
                'measured_qubit': qubit_index,
                'measurement_gate': 'M',
                'classical_bit': clbit_index
            }
    return measurement_info


def add_measurements_from_info(circuit, measurement_info, exclude_qk=True):
    qk_index = None
    if exclude_qk and len(circuit.qregs) > 0:
        qk_index = circuit.qubits.index(circuit.qregs[-1][0])

    for qubit, info in measurement_info.items():
        qubit_index = info['measured_qubit']
        clbit_index = info['classical_bit']
        if exclude_qk and qubit_index == qk_index:
            continue
        circuit.measure(qubit_index, clbit_index)
    return circuit

In [ ]:
def key_phase(key):
    match key:
        case '000': return 0.25 * pi
        case '001': return 0.5 * pi
        case '010': return 0.75 * pi
        case '011': return 1.0 * pi
        case '100': return 1.25 * pi
        case '101': return 1.5 * pi
        case '110': return 1.75 * pi
        case '111': return 2.0 * pi


def print_qk_status(stage_name, circuit, key=None, real_gate_indices=None):
    print(f"\n{'='*50}")
    print(f"STAGE: {stage_name}")
    print(f"{'='*50}")
    qk_reg = next((r for r in circuit.qregs if r.name == 'qk'), None)
    print(f"qk present    : {qk_reg is not None}")
    print(f"Num qubits    : {circuit.num_qubits}")
    print(f"Registers     : {circuit.qregs}")
    print(f"Gate counts   : {circuit.count_ops()}")
    if qk_reg is not None:
        qk = qk_reg[0]
        qk_gates = [inst for inst in circuit.data if qk in inst.qubits]
        print(f"Gates on qk   : {len(qk_gates)}")
        for inst in qk_gates:
            print(f"  {inst.operation.name} on "
                  f"{[circuit.find_bit(q).index for q in inst.qubits]}")
    if key is not None:
        print(f"Key           : {key}")
        print(f"Real indices  : {real_gate_indices}")
    qk_meas = [inst for inst in circuit.data
                if inst.operation.name == 'measure'
                and qk_reg is not None
                and qk_reg[0] in inst.qubits]
    print(f"qk measured   : {len(qk_meas) > 0} ({len(qk_meas)} measurements)")


def total_variation_distance_from_counts(counts1, counts2, shots=1024, 
                                          plot=True, labels=None):
    if labels is None:
        labels = ["Distribution 1", "Distribution 2"]
    all_states = set(counts1.keys()).union(set(counts2.keys()))
    b = len(next(iter(all_states)))
    state_to_index = {state: i for i, state in enumerate(all_states)}
    p = np.zeros(len(all_states))
    q = np.zeros(len(all_states))
    for state, count in counts1.items():
        p[state_to_index[state]] = count
    for state, count in counts2.items():
        q[state_to_index[state]] = count
    tvd = np.sum(np.abs(p - q)) / (2 * shots * (2 ** b))
    if plot:
        fig = plot_histogram([counts1, counts2], legend=labels, figsize=(12, 6))
        display(fig)
    return tvd

In [ ]:
# Create test circuit
qr = QuantumRegister(2, 'q')
cr = ClassicalRegister(2, 'c')
test_qc = QuantumCircuit(qr, cr)
test_qc.h(0)
test_qc.cx(0, 1)
test_qc.measure(0, 0)
test_qc.measure(1, 1)

print("=== Original Test Circuit ===")
print(f"Gate counts: {test_qc.count_ops()}")
test_qc.draw('mpl')

In [ ]:
# Step 1: Simulate original
transpiled_test = transpile(test_qc, backend)
counts_test_original = backend.run(transpiled_test, shots=1024).result().get_counts()
print(f"Original counts: {counts_test_original}")

# Step 2: Lock
test_locked, test_key, test_info = transform_circuit(test_qc, seed=42)
print(f"\nKey: {test_key}")
print(f"Real gates: {test_info['real_gate_indices']}")
print_qk_status("After locking", test_locked, test_key, test_info['real_gate_indices'])

# Step 3: Apply phase wall
test_p_offset = key_phase("010")
test_locked = modify_rz_inside_barriers(test_locked, -test_p_offset)
print_qk_status("After phase wall", test_locked)

# Step 4: Decrypt
test_decrypted = decrypt_full_circuit(
    locked_circuit=test_locked,
    qll_key=test_key,
    phase_offset=test_p_offset
)
print_qk_status("After decryption", test_decrypted)

# Step 5: Gate comparison
print("\n=== Gate comparison ===")
orig_instrs = [inst for inst in test_qc.data
               if inst.operation.name not in ('barrier', 'measure')]
dec_instrs = [inst for inst in test_decrypted.data
              if inst.operation.name not in ('barrier', 'measure')]

print(f"{'#':<4} {'Original':<20} {'Decrypted':<20} {'Match'}")
for i, (o, d) in enumerate(zip(orig_instrs, dec_instrs)):
    o_q = [test_qc.find_bit(q).index for q in o.qubits]
    d_q = [test_decrypted.find_bit(q).index for q in d.qubits]
    match = "✓" if o.operation.name == d.operation.name and o_q == d_q else "✗ DIFF"
    print(f"{i:<4} {o.operation.name+str(o_q):<20} {d.operation.name+str(d_q):<20} {match}")

if len(orig_instrs) != len(dec_instrs):
    print(f"\nLength mismatch! Original: {len(orig_instrs)}, Decrypted: {len(dec_instrs)}")

# Step 6: Simulate decrypted
test_dec_clean = remove_all_measurements(test_decrypted)
test_meas_info = get_measurement_info(test_qc, exclude_qk=False)
test_dec_clean = add_measurements_from_info(test_dec_clean, test_meas_info, exclude_qk=True)

transpiled_test_dec = transpile(test_dec_clean, backend)
counts_test_decrypted = backend.run(transpiled_test_dec, shots=1024).result().get_counts()

print(f"\nOriginal counts : {counts_test_original}")
print(f"Decrypted counts: {counts_test_decrypted}")

# Step 7: TVD
tvd = total_variation_distance_from_counts(
    counts_test_original,
    counts_test_decrypted,
    shots=1024,
    plot=True,
    labels=["Original", "Decrypted"]
)
print(f"TVD: {tvd:.6f}")